# YOLO i klasyfikacja obiektów na obrazach
##### Celem niniejszego notatnika jest analiza architektury YOLO w zadaniu klasyfikacji obiektów na obrazach.

##### W pierwszej kolejności porównano popularne wersje modelu YOLO z ostatnich lat — YOLOv5s, YOLOv8s oraz YOLO11s — wykorzystując ten sam zbiór danych zawierający elementy garderoby.

##### Następnie przeanalizowano skuteczność najnowszej wersji modelu, YOLO26, udostępnionej w styczniu 2026 roku, w rozwiązaniu tego samego problemu.

##### Dodatkowo zbadano wpływ wariantów Nano (n), Small (s) oraz Medium (m) na proces uczenia modelu, jego dokładność oraz wymagania obliczeniowe.

In [1]:
import os
import shutil
import random
import yaml
import kagglehub

path = kagglehub.dataset_download("nguyngiabol/colorful-fashion-dataset-for-object-detection")
print(f"Ścieżka bazowa Kaggle: {path}")
potencjalny_folder = os.path.join(path, "colorful_fashion_dataset_for_object_detection")

if os.path.exists(potencjalny_folder):
    src_images = os.path.join(potencjalny_folder, "JPEGImages")
    src_labels = os.path.join(potencjalny_folder, "Annotations_txt")
else:
    src_images = os.path.join(path, "JPEGImages")
    src_labels = os.path.join(path, "Annotations_txt")

print(f"Wykryta ścieżka do zdjęć: {src_images}")
print(f"Wykryta ścieżka do etykiet: {src_labels}")

yolo_dataset_path = os.path.abspath("yolo_fashion_dataset")
folders = [
    "images/train", "images/val",
    "labels/train", "labels/val"
]

for folder in folders:
    os.makedirs(os.path.join(yolo_dataset_path, folder), exist_ok=True)

wszystkie_zdjecia = [f for f in os.listdir(src_images) if f.endswith('.jpg')]
random.shuffle(wszystkie_zdjecia)
# Podział 80% trening, 20% walidacja
split_index = int(len(wszystkie_zdjecia) * 0.8)
train_files = wszystkie_zdjecia[:split_index]
val_files = wszystkie_zdjecia[split_index:]

# Kopiowanie plików do nowych folderów
def kopiuj_pliki(lista_plikow, split_name):
    print(f"Kopiowanie danych dla: {split_name}...")
    for plik in lista_plikow:
        # Kopiowanie zdjęcia
        shutil.copy(
            os.path.join(src_images, plik),
            os.path.join(yolo_dataset_path, f"images/{split_name}", plik)
        )
        # Kopiowanie odpowiadającej etykiety .txt
        nazwa_txt = plik.replace('.jpg', '.txt')
        if os.path.exists(os.path.join(src_labels, nazwa_txt)):
            shutil.copy(
                os.path.join(src_labels, nazwa_txt),
                os.path.join(yolo_dataset_path, f"labels/{split_name}", nazwa_txt)
            )

kopiuj_pliki(train_files, "train")
kopiuj_pliki(val_files, "val")

# 5. Tworzenie pliku data.yaml
yaml_content = {
    'path': yolo_dataset_path,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 10,
    'names': ['sunglass', 'hat', 'jacket', 'shirt', 'pants', 'shorts', 'skirt', 'dress', 'bag', 'shoe']
}

yaml_file_path = os.path.join(yolo_dataset_path, 'data.yaml')
with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"Dataset został sformatowany.")
print(f"Plik konfiguracyjny do treningu YOLO to: {yaml_file_path}")

Ścieżka bazowa Kaggle: /kaggle/input/datasets/nguyngiabol/colorful-fashion-dataset-for-object-detection
Wykryta ścieżka do zdjęć: /kaggle/input/datasets/nguyngiabol/colorful-fashion-dataset-for-object-detection/colorful_fashion_dataset_for_object_detection/JPEGImages
Wykryta ścieżka do etykiet: /kaggle/input/datasets/nguyngiabol/colorful-fashion-dataset-for-object-detection/colorful_fashion_dataset_for_object_detection/Annotations_txt
Kopiowanie danych dla: train...
Kopiowanie danych dla: val...
Dataset został sformatowany.
Plik konfiguracyjny do treningu YOLO to: /kaggle/working/yolo_fashion_dataset/data.yaml


In [2]:
!pip install ultralytics
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.0 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numb

In [3]:
import torch
print("Czy GPU jest dostępne?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nazwa karty:", torch.cuda.get_device_name(0))

Czy GPU jest dostępne?: True
Nazwa karty: Tesla P100-PCIE-16GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


Ewolucja YOLO – Porównanie wersji Small (v5s, v8s, 11s)

In [3]:
from ultralytics import YOLO

yaml_path = 'yolo_fashion_dataset/data.yaml'
epochs_count = 30

# 1. YOLOv5 Small
print("--- Trening YOLOv5s ---")
model_v5 = YOLO('yolov5s.pt')
model_v5.train(data=yaml_path, epochs=epochs_count, imgsz=640, project="Ewolucja_YOLO", name="YOLOv5s")

# 2. YOLOv8 Small
print("--- Trening YOLOv8s ---")
model_v8 = YOLO('yolov8s.pt')
model_v8.train(data=yaml_path, epochs=epochs_count, imgsz=640, project="Ewolucja_YOLO", name="YOLOv8s")

# 3. YOLO11 Small
print("--- Trening YOLO11s ---")
model_11 = YOLO('yolo11s.pt')
model_11.train(data=yaml_path, epochs=epochs_count, imgsz=640, project="Ewolucja_YOLO", name="YOLO11s")

--- Trening YOLOv5s ---
PRO TIP 💡 Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_fashion_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e9018982060>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [7]:
import os
import shutil
from google.colab import files

zip_name = "wyniki_projektu_YOLO"
if os.path.exists("runs"):
    print("Znaleziono folder 'runs'.")

    shutil.make_archive(zip_name, 'zip', root_dir="runs")
    files.download(f"{zip_name}.zip")
else:
    print("Nie znaleziono folderu 'runs'.")

Znaleziono folder 'runs'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Strojenie YOLOv8 (Nano\Small\Medium) + Augmentacja

In [3]:
from ultralytics import YOLO

yaml_path = 'yolo_fashion_dataset/data.yaml'
epochs_count = 30

# 1. YOLOv8 Nano
print("--- Trening YOLOv8 Nano ---")
model_v8n = YOLO('yolov8n.pt')
model_v8n.train(data=yaml_path, epochs=epochs_count, imgsz=640, project="Strojenie_YOLOv8", name="YOLOv8_Nano", device=0)

# 2. YOLOv8 Medium
print("--- Trening YOLOv8 Medium ---")
model_v8m = YOLO('yolov8m.pt')
model_v8m.train(data=yaml_path, epochs=epochs_count, imgsz=640, project="Strojenie_YOLOv8", name="YOLOv8_Medium", device=0)

# 3. YOLOv8 Small + Agresywna Augmentacja ubrań
print("--- Trening YOLOv8 Small z Augmentacją ---")
model_v8s_aug = YOLO('yolov8s.pt')
model_v8s_aug.train(
    data=yaml_path,
    epochs=epochs_count,
    imgsz=640,
    project="Strojenie_YOLOv8",
    name="YOLOv8_Small_Aug",
    device=0, 
    # Parametry augmentacji:
    degrees=20.0,   # Losowa rotacja zdjęć ubrań o  ~20 stopni
    scale=0.5,      # Losowe skalowanie (przybliżanie/oddalanie)
    flipud=0.5,     # Obracanie góra-dół
    mixup=0.2       # Nakładanie dwóch ubrań na siebie z przezroczystością
)

--- Trening YOLOv8 Nano ---
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_fashion_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv8_Nano, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a3062822ed0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

Najnowsza wersja – YOLO26 (Styczeń 2026)

In [4]:
model_26 = YOLO('yolo26s.pt')
model_26.train(data=yaml_path, epochs=20, imgsz=640, project="Najnowsze_YOLO", name="YOLO26s")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_fashion_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO26s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2f2df20770>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [6]:
import shutil

shutil.make_archive(
    "/kaggle/working/runs_backup",
    "zip",
    "/kaggle/working/runs"
)

'/kaggle/working/runs_backup.zip'

In [9]:
!find /kaggle/working/runs -name "best.pt"

/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Nano/weights/best.pt
/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Small_Aug/weights/best.pt
/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Medium/weights/best.pt
/kaggle/working/runs/detect/Najnowsze_YOLO/YOLO26s/weights/best.pt


Zilustrowanie uzyskanych wyników

In [12]:
import os
from ultralytics import YOLO


def find_images():
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                return root
    return None

test_dir = find_images()

print("Znaleziony folder zdjęć:", test_dir)

if test_dir is None:
    raise Exception("Nie znaleziono zdjęć w /kaggle/input")

models = {
    "YOLOv8_Nano": "/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Nano/weights/best.pt",
    "YOLOv8_Small_Aug": "/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Small_Aug/weights/best.pt",
    "YOLOv8_Medium": "/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Medium/weights/best.pt",
    "YOLO26s": "/kaggle/working/runs/detect/Najnowsze_YOLO/YOLO26s/weights/best.pt",
}

# Predykcja

output_dir = "/kaggle/working/predictions"

for name, path in models.items():
    print(f"\nModel: {name}")

    model = YOLO(path)

    model.predict(
        source=test_dir,
        save=True,
        conf=0.25,
        project=output_dir,
        name=name
    )

print("\nWyniki:", output_dir)

Znaleziony folder zdjęć: /kaggle/input/datasets/juliazverko/test-photos

Model: YOLOv8_Nano

image 1/7 /kaggle/input/datasets/juliazverko/test-photos/01.jpg: 384x640 1 shoe, 7.1ms
image 2/7 /kaggle/input/datasets/juliazverko/test-photos/02.jpg: 640x480 1 shirt, 1 pants, 1 shoe, 6.7ms
image 3/7 /kaggle/input/datasets/juliazverko/test-photos/03.jpg: 640x384 1 jacket, 1 pants, 1 shoe, 6.7ms
image 4/7 /kaggle/input/datasets/juliazverko/test-photos/04.jpg: 640x448 1 shirt, 6.6ms
image 5/7 /kaggle/input/datasets/juliazverko/test-photos/05.jpg: 640x448 (no detections), 6.1ms
image 6/7 /kaggle/input/datasets/juliazverko/test-photos/06.jpg: 448x640 2 shoes, 6.7ms
image 7/7 /kaggle/input/datasets/juliazverko/test-photos/07.jpg: 448x640 1 shirt, 6.1ms
Speed: 1.8ms preprocess, 6.6ms inference, 1.1ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /kaggle/working/predictions/YOLOv8_Nano-2

Model: YOLOv8_Small_Aug

image 1/7 /kaggle/input/datasets/juliazverko/test-photos/01.jpg: 384

# Metryki

In [17]:
import time
import pandas as pd
from ultralytics import YOLO

test_dir = "/kaggle/input/datasets/juliazverko/test-photos"

models = {
    "YOLOv8_Nano": "/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Nano/weights/best.pt",
    "YOLOv8_Small_Aug": "/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Small_Aug/weights/best.pt",
    "YOLOv8_Medium": "/kaggle/working/runs/detect/Strojenie_YOLOv8/YOLOv8_Medium/weights/best.pt",
    "YOLO26s": "/kaggle/working/runs/detect/Najnowsze_YOLO/YOLO26s/weights/best.pt",
}

results = []

for name, path in models.items():
    print(f"\n================ {name} ================")

    model = YOLO(path)
    # precision, recall, mAP
    metrics = model.val(data="/kaggle/working/yolo_fashion_dataset/data.yaml")

    precision = metrics.box.mp
    recall = metrics.box.mr
    map50 = metrics.box.map50
    map5095 = metrics.box.map

    # czas inferencji
    start = time.time()

    preds = model.predict(
        source=test_dir,
        save=False,
        conf=0.25
    )

    end = time.time()

    total_images = len(preds)
    inference_time = (end - start) / total_images

    results.append({
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "mAP50": map50,
        "mAP50-95": map5095,
        "Inference_sec_per_image": inference_time
    })

df = pd.DataFrame(results)
df


================ YOLOv8_Nano ================
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1446.5±415.0 MB/s, size: 39.5 KB)
val: Scanning /kaggle/working/yolo_fashion_dataset/labels/val.cache... 537 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 537/537 173.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 7.1it/s 4.8s0.1s
                   all        537       2065      0.756       0.77       0.79      0.528
              sunglass         82         82      0.579      0.252      0.429      0.135
                   hat         73         73      0.702      0.781      0.756      0.395
                jacket        182        186      0.759      0.823      0.832      0.649
                 shirt        367        373      0.777      0.866  

,Model,Precision,Recall,mAP50,mAP50-95,Inference_sec_per_image
0,YOLOv8_Nano,0.756036,0.770134,0.790052,0.528496,0.028843
1,YOLOv8_Small_Aug,0.786358,0.773432,0.804283,0.528313,0.034652
2,YOLOv8_Medium,0.784123,0.774181,0.810486,0.550809,0.048878
3,YOLO26s,0.784687,0.767976,0.798993,0.536279,0.033761
